In [1]:
from sklearn.datasets import fetch_openml
import numpy as np
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
mnist.target = mnist.target.astype(np.uint8)
X = mnist["data"]
y = mnist["target"]

In [2]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

kmeans_sil = []
for k in range(6, 13):
    kmeans = KMeans(n_clusters=k, n_init=10)
    labels = kmeans.fit_predict(X)
    if k == 10:
        y_pred10 = labels
    sil = silhouette_score(X, labels)
    kmeans_sil.append(float(sil))
    print(f"Wskaźnik sylwetkowy dla k={k}: {sil}")

Wskaźnik sylwetkowy dla k=6: 0.06539304614283295
Wskaźnik sylwetkowy dla k=7: 0.06664431952913571
Wskaźnik sylwetkowy dla k=8: 0.07342496548747358
Wskaźnik sylwetkowy dla k=9: 0.056817743861047855
Wskaźnik sylwetkowy dla k=10: 0.058639852351611524
Wskaźnik sylwetkowy dla k=11: 0.05834348905464587
Wskaźnik sylwetkowy dla k=12: 0.05901412733245104


In [3]:
import pickle

with open('kmeans_sil.pkl', 'wb') as f:
    pickle.dump(kmeans_sil, f)

In [4]:
from sklearn.metrics import confusion_matrix

cmx = confusion_matrix(y, y_pred10)
print(cmx)

[[  71   38 5063    1    9  292 1250    4    7  168]
 [   8    6    0 4293   10    8    7 3527   11    7]
 [ 199  215   57  424 4866  320  250  435   79  145]
 [1069  189   24  449  217 4595  466   55   46   31]
 [  18 3727    9  179   29    0  308  224 2164  166]
 [1155  426   60  155    7 2132 1843  258  209   68]
 [  15   67   73  190   55   38 2044   43    4 4347]
 [  19 2090   21  372   53    5   13  310 4406    4]
 [4114  211   37  333   54 1208  325  305  189   49]
 [  89 3454   51  263   20   89   31   91 2854   16]]


In [5]:
argmax_indices = np.argmax(cmx, axis=1)
argmax_indices_list = sorted(list(set(int(x) for x in argmax_indices)))

with open('kmeans_argmax.pkl', 'wb') as f:
    pickle.dump(argmax_indices_list, f)

In [6]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=2, metric='euclidean', n_jobs=-1)
nn.fit(X)

distances, indices = nn.kneighbors(X[:300])

min_distances = []
for dists in distances:
    non_zero = dists[dists > 0]
    if len(non_zero) > 0:
        min_distances.append(float(non_zero[0]))

top_10 = sorted(min_distances)[:10]
print(f"10 najmniejszych odległości: {top_10}")
with open('dist.pkl', 'wb') as f:
    pickle.dump(top_10, f)

10 najmniejszych odległości: [279.26152617215286, 304.37641170103836, 317.5893575043093, 352.89800226127664, 355.1774204534967, 359.64287842247063, 360.42474942767177, 379.2110230465354, 387.63642759678817, 387.9587606949996]


In [7]:
from sklearn.cluster import DBSCAN

s = np.mean(min_distances)
eps_values = [s, s + 0.25 * s, s + 0.50 * s, s + 0.75 * s]

unique_labels_counts = []

for eps_val in eps_values:
    dbscan = DBSCAN(eps=eps_val, min_samples=5)
    dbscan.fit(X)
    num_unique_labels = len(set(dbscan.labels_))
    unique_labels_counts.append(int(num_unique_labels))
    print(f"Dla eps={eps_val:.4f} znaleziono {num_unique_labels} unikalnych etykiet.")

with open('dbscan_len.pkl', 'wb') as f:
    pickle.dump(unique_labels_counts, f)

Dla eps=1015.6297 znaleziono 152 unikalnych etykiet.
Dla eps=1269.5371 znaleziono 102 unikalnych etykiet.
Dla eps=1523.4446 znaleziono 15 unikalnych etykiet.
Dla eps=1777.3520 znaleziono 3 unikalnych etykiet.


In [8]:
from collections import Counter

dbscan = DBSCAN(eps=1523.4446, min_samples=5, n_jobs=-1)
dbscan.fit(X)

cluster_distribution = Counter(dbscan.labels_)
for label, count in sorted(cluster_distribution.items()):
    if label == -1:
        print(f" -> Szum (etykieta -1): {count} instancji")
    else:
        print(f" -> Klaster {label}: {count} instancji")

 -> Szum (etykieta -1): 4482 instancji
 -> Klaster 0: 65440 instancji
 -> Klaster 1: 7 instancji
 -> Klaster 2: 15 instancji
 -> Klaster 3: 4 instancji
 -> Klaster 4: 3 instancji
 -> Klaster 5: 5 instancji
 -> Klaster 6: 8 instancji
 -> Klaster 7: 5 instancji
 -> Klaster 8: 5 instancji
 -> Klaster 9: 3 instancji
 -> Klaster 10: 6 instancji
 -> Klaster 11: 5 instancji
 -> Klaster 12: 4 instancji
 -> Klaster 13: 8 instancji
